In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
import random
import time
from IPython.display import display, HTML

##Load the base configuration (Engine Model)
import sys
sys.path.append(r'C:\Users\Public\Documents\CIF\orchestrations\Python')
import cif_orchestration_base
import time

#Set the IP address and port of the CIF manager on the target
server_ip = "192.168.1.222"
manager_port = "15882"

#Configuration for plugins including channel links  
engine_model = cif_orchestration_base.plugin("engine_model", "Model_Interface_Plugin", "", "")
tag_monitor = cif_orchestration_base.plugin("tagmon", "Tag_Monitor_Plugin", "", "")

#Create connection to the CIF manager
cif_manager = cif_orchestration_base.cif_manager(server_ip, manager_port)

#Business logic (Load plugins, link channel, configure plugins, change state, etc)
cif_manager.load(engine_model)
cif_manager.load(tag_monitor)

# This forces the Jupyter cell container to occupy 100% of the screen width
display(HTML("<style>.container { width:100% !important; } .jp-OutputArea-output { width:100% !important; }</style>"))


# 1. Setup the Plotly Figure (Subplots: Gauge on left, Line chart on right)
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'indicator'}, {'type': 'xy'}]],
    column_widths=[1, 1]
)

# Add the Gauge (Dial)
fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=0,
    title={'text': "Engine RPM"},
    gauge={'axis': {'range': [0, 7000]}, 'bar': {'color': "darkblue"}},
    domain={'x': [0, 1], 'y': [0, 1]}
), row=1, col=1)

# Add the Line Chart
fig.add_trace(go.Scatter(x=[], y=[], mode='lines+markers', name='History'), row=1, col=2)

fig.update_layout(height=400, template='plotly_dark', showlegend=False)

# fig.update_layout(
#     height=600,             # Keep this high for vertical presence
#     # autosize=True,          # Tells Plotly to fill the container
#     margin=dict(l=10, r=10, t=50, b=10), 
#     template='plotly_dark'
# )

# Display with a layout that forces 100% width on the widget itself
fw = go.FigureWidget(fig)
fw.layout.width = None # Let the CSS control the width
# display(widgets.VBox([fw, stop_button], layout=widgets.Layout(width='100%')))


# 2. Create the Widget Container
# We use FigureWidget for "in-place" updates without flickering
# fw = go.FigureWidget(fig)

# 3. Control UI (Stop Button)
stop_button = widgets.Button(
    description='Shut Down Dashboard',
    button_style='danger',
    icon='power-off'
)

is_running = True
def on_stop_clicked(b):
    global is_running
    is_running = False
    stop_button.description = "System Offline"
    stop_button.disabled = True

stop_button.on_click(on_stop_clicked)

# 4. Display the Dashboard
display(widgets.VBox([fw, stop_button]))

# 5. Live Update Loop
history = []
timestamps = []
counter = 0

try:
    while is_running:
        # Generate new data
        xil_read = tag_monitor.monitor_tag(["engine_model.RPM"])
        new_val = (xil_read.tag_values[0])
        history.append(new_val)
        timestamps.append(counter)
        
        # Keep only last 30 points for the chart
        if len(history) > 30:
            history.pop(0)
            timestamps.pop(0)
        
        # Update the Gauge (Trace 0)
        fw.data[0].value = new_val
        
        # Update the Line Chart (Trace 1)
        fw.data[1].x = timestamps
        fw.data[1].y = history
        
        counter += 1
        time.sleep(0.3) # Fast updates

except KeyboardInterrupt:
    is_running = False


    'data': [{'domain': {'x': [0.0, 0.45], 'y': [0, 1]},
              'gauge': …